# COCO Captions 2014 — hardcore

Model scratch **chưa train**; dự đoán chỉ minh họa inference, không phải kết quả chất lượng. Bạn tự thiết kế tiền xử lý, thay model và viết training sau khi đọc luồng vào/ra này.

## 1. Load dữ liệu cơ bản

Tập inference: **phần val2014 ngoài 5000 ảnh validation seed=42, đúng giao thức hiện có; không phải COCO hidden test**. Nguồn/revision trong SOURCE/DATASET_ID. Dữ liệu thô được giữ nguyên; thao tác nhóm/sắp ID chỉ để giữ giao thức mẫu của hai bản đầy đủ, không fit đặc trưng.

Chạy từ thư mục dự án, sửa DATA_DIR trước Run All. DATA_MODE="native" đọc nguồn đã khai báo; DATA_MODE="local_jsonl" đọc bản xuất **đúng nguồn**, schema: val2014.jsonl: image_id:int, image:path; xuất toàn bộ val2014 theo nguồn COCO2014. Đường dẫn ảnh trong JSONL phải hợp lệ từ thư mục chạy. Nếu dùng nguồn HF cần cài datasets; COCO cần torchvision/pycocotools; TFDS cần kernel TensorFlow tương thích. Không tự thay dataset khi thiếu dữ liệu.

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import random
import string
import unicodedata
import numpy as np
import torch
from torch import nn
from PIL import Image

SEED=42
BATCH_SIZE=16
MAX_INPUT_BYTES=4096  # Giới hạn đầu vào cố định của model; không đổi số mẫu inference.
MAX_OUTPUT_TOKENS=32
IMAGE_SIZE=64
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DATA_MODE="native"  # Chọn local_jsonl rõ ràng để dùng bản xuất đúng dataset/schema bên dưới.

DATA_DIR=Path('data/coco2014')
CACHE_DIR=Path("artifacts/caption_coco2014/hardcore/data_cache")
INFERENCE_SPLIT='held_out_val2014'
SOURCE={'row': 2, 'task': 'Image Captioning', 'dataset': 'MS COCO Captions 2014', 'url': 'https://docs.pytorch.org/vision/master/generated/torchvision.datasets.CocoCaptions.html', 'reader': 'torchvision.datasets.CocoCaptions', 'split_protocol': 'train2014; val2014 chia image-disjoint validation5000/held-out còn lại'}
DATASET_ID='MS COCO Captions 2014'
DATASET_REVISION=None
VALIDATION_IMAGE_COUNT=5000


def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path}. Cung cấp JSONL từ đúng dataset/revision/schema đã ghi; không dùng dữ liệu giả thay thế.")
    with path.open(encoding="utf-8") as handle:
        rows=[json.loads(line) for line in handle if line.strip()]
    if not rows:raise ValueError(f"Tập dữ liệu rỗng: {path}")
    return rows

In [ ]:
if DATA_MODE=="local_jsonl":
    # Bản xuất phải giữ toàn bộ val2014, không xuất riêng một subset rồi chia lại.
    rows=read_jsonl(DATA_DIR/"val2014.jsonl")
    rows=sorted(rows,key=lambda row:int(row["image_id"]))
    records=[{"id":f"val2014:{int(row['image_id'])}","image":row["image"],"image_id":int(row["image_id"])} for row in rows]
elif DATA_MODE=="native":
    from torchvision.datasets import CocoCaptions
    image_root=DATA_DIR/"val2014"
    annotation=DATA_DIR/"annotations/captions_val2014.json"
    if not image_root.is_dir() or not annotation.is_file():raise FileNotFoundError("Cần data/coco2014/val2014 và annotations/captions_val2014.json từ COCO2014; CocoCaptions không tự tải ảnh.")
    dataset=CocoCaptions(root=str(image_root),annFile=str(annotation))
    records=[{"id":f"val2014:{image_id}","image_id":image_id,"image":str(image_root/dataset.coco.loadImgs(image_id)[0]["file_name"])} for image_id in sorted(dataset.ids)]
else:raise ValueError("DATA_MODE phải là native hoặc local_jsonl.")
if len(records)<=VALIDATION_IMAGE_COUNT:raise ValueError("Không đủ toàn bộ val2014 để chia giao thức đã ghi; kiểm tra dữ liệu.")
order=list(range(len(records)));random.Random(SEED).shuffle(order)
val_ids=set(order[:VALIDATION_IMAGE_COUNT])
records=[row for index,row in enumerate(records) if index not in val_ids]

assert records, "Không có mẫu test/validation để inference."
assert len({row["id"] for row in records})==len(records), "ID đầu vào bị trùng."
print("Tập inference:",INFERENCE_SPLIT,"; số mẫu:",len(records))

## 2. Định nghĩa một model và inference trực tiếp

Các lớp học đều khởi tạo ngẫu nhiên. Chỉ chuyển đầu vào tối thiểu bên trong model; không có tokenizer/vocabulary fit, hàm tiền xử lý riêng, loss, optimizer, training hay checkpoint. Giới hạn chiều đầu vào/đầu ra ở phần cấu hình là cố định và chỉ phục vụ model minh họa; toàn bộ mẫu của split vẫn được inference. Không truyền gold target vào forward.

In [ ]:
class RawImageTextModel(nn.Module):
    """CNN và decoder scratch; nhận ảnh thô, tùy bài có thêm câu hỏi."""
    def __init__(self):
        super().__init__()
        self.visual=nn.Sequential(nn.Conv2d(3,8,3,2,1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(8,32))
        self.embedding=nn.Embedding(257,16)
        self.question=nn.Linear(16,32)
        self.decoder=nn.RNNCell(16,32)
        self.head=nn.Linear(32,96)
    def forward(self,rows):
        outputs=[]
        for row in rows:
            value=row["image"]
            if isinstance(value,dict):
                value=Image.open(io.BytesIO(value["bytes"])) if value.get("bytes") is not None else Image.open(value["path"])
            elif isinstance(value,(str,Path)):value=Image.open(value)
            elif not isinstance(value,Image.Image):raise TypeError("Ảnh phải là PIL, path hoặc HF image bytes/path.")
            # Chuyển đầu vào cơ học nằm trong model, không có transform pipeline bên ngoài.
            array=np.array(value.convert("RGB").resize((IMAGE_SIZE,IMAGE_SIZE)),copy=True)
            tensor=torch.from_numpy(array).permute(2,0,1).float()[None].to(self.head.weight.device)/255
            state=self.visual(tensor)
            if "question" in row:
                ids=list(str(row["question"]).encode("utf-8")[:MAX_INPUT_BYTES]) or [256]
                state=state+self.question(self.embedding(torch.tensor(ids,device=tensor.device)).mean(0,keepdim=True))
            token=torch.tensor([256],device=tensor.device)
            text=[]
            for _ in range(MAX_OUTPUT_TOKENS):
                state=self.decoder(self.embedding(token),state)
                index=int(self.head(state).argmax(-1).item())
                if index==95:break
                text.append(chr(32+index))
                token=torch.tensor([32+index],device=tensor.device)
            outputs.append("".join(text))
        return outputs

In [ ]:
model=RawImageTextModel().to(DEVICE)
model.eval()
predictions=[]
with torch.inference_mode():
    for start in range(0,len(records),BATCH_SIZE):
        batch=records[start:start+BATCH_SIZE]
        inputs=[{"image":row["image"],**({"question":row["question"]} if "question" in row else {})} for row in batch]
        outputs=model(inputs)
        assert len(outputs)==len(batch), "Số dự đoán trong batch không khớp."
        predictions.extend([{'id':row["id"],'prediction':output} for row,output in zip(batch,outputs)])

assert len(predictions)==len(records)>0
assert [row['id'] for row in predictions]==[row["id"] for row in records]
assert len({row['id'] for row in predictions})==len(records)
assert all(isinstance(row['prediction'],str) for row in predictions)

print(f"Đã inference {len(predictions)} mẫu trên {INFERENCE_SPLIT} bằng model scratch CHƯA TRAIN.")
print("Đầu ra minh họa thực thi, chưa đo chất lượng; cần tự xây dựng tiền xử lý, model và training.")
print(predictions[:2])
# File dự đoán chỉ được tạo khi người dùng chạy notebook; không cần checkpoint/helper.
PREDICTION_PATH=Path("artifacts/caption_coco2014/hardcore/predictions.json")
PREDICTION_PATH.parent.mkdir(parents=True,exist_ok=True)
PREDICTION_PATH.write_text(json.dumps(predictions,ensure_ascii=False,indent=2),encoding="utf-8")